In [17]:
# Block 1: Dependencies Installation
!pip install -qU langgraph langchain-core langchain-community transformers accelerate

In [18]:
# Block 2: Imports & Open-Source LLM Pipeline Setup
import operator
from typing import TypedDict, Annotated
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, START, END

# Free Local Open-Source Model Load Karein
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.01,  # Lower temperature stops model hallucination
    do_sample=False,   # Deterministic output
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:
# Block 3: State Definition
# Is State Object mein shipment input, 3 parallel risk factors (Weather, Traffic, Cost),
# combine reports list, aur final risk assessment store hoga.

class ShipmentRiskState(TypedDict):
    shipment_info: str
    weather_risk: str
    traffic_risk: str
    cost_risk: str
    overall_assessment: str
    # Parallel nodes ke individual outputs ko aggregate karne ke liye List Reducer
    risk_reports: Annotated[list, operator.add]

In [20]:
# Block 4 (Fixed with Few-Shot Examples & Clean Output)

def node_weather_risk(state: ShipmentRiskState):
    prompt = (
        f"<|user|>\n"
        f"You are a Weather Risk Agent. Evaluate ONLY weather conditions (rain, snow, ice, temperature, fog).\n"
        f"Ignore all traffic, delays, or cost information.\n\n"
        f"Example:\n"
        f"Data: 'Sunny 75F, Heavy traffic jam.'\n"
        f"Output: Risk: Low. Weather is clear (75F).\n\n"
        f"Shipment Data: '{state['shipment_info']}'\n"
        f"<|assistant|>\n"
    )
    res = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "weather_risk": res,
        "risk_reports": [f"[WEATHER RISK]\n{res}"]
    }

def node_traffic_risk(state: ShipmentRiskState):
    prompt = (
        f"<|user|>\n"
        f"You are a Traffic Risk Agent. Evaluate ONLY road congestion, accidents, construction, and delays.\n"
        f"Ignore weather conditions and financial costs.\n\n"
        f"Example:\n"
        f"Data: 'Rainy weather, 15-mile traffic jam due to construction.'\n"
        f"Output: Risk: High. Severe traffic jam caused by construction.\n\n"
        f"Shipment Data: '{state['shipment_info']}'\n"
        f"<|assistant|>\n"
    )
    res = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "traffic_risk": res,
        "risk_reports": [f"[TRAFFIC RISK]\n{res}"]
    }

def node_cost_risk(state: ShipmentRiskState):
    prompt = (
        f"<|user|>\n"
        f"You are a Financial Risk Agent. Evaluate ONLY financial risks (fuel surcharges, tolls, late penalties, theft).\n"
        f"If no explicit financial risks are mentioned, output 'Risk: Low. No financial surcharges or penalties specified.'\n\n"
        f"Shipment Data: '{state['shipment_info']}'\n"
        f"<|assistant|>\n"
    )
    res = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "cost_risk": res,
        "risk_reports": [f"[COST RISK]\n{res}"]
    }

In [21]:
# Block 5: Define Risk Assessment Agent (Aggregator Node)

def node_risk_assessment(state: ShipmentRiskState):
    """
    Aggregator Node: Teeno parallel agents (Weather, Traffic, Cost) ke results ko combine
    karke overall risk score, report, aur recommendations generate karta hai.
    """
    combined_risks = "\n\n".join(state["risk_reports"])

    prompt = (
        f"<|user|>\n"
        f"Shipment Information:\n{state['shipment_info']}\n\n"
        f"Evaluated Risk Factors:\n{combined_risks}\n\n"
        f"Based on the above factors, synthesize an Overall Risk Assessment Report. "
        f"Include an overall Risk Level (Low/Medium/High) and 2-3 actionable Recommendations.\n"
        f"<|assistant|>\n"
    )

    response = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "overall_assessment": response,
        "risk_reports": [f"[FINAL RISK REPORT & RECOMMENDATIONS]\n{response}"]
    }

In [22]:
# Block 6: LangGraph Construction (Parallel Fan-Out & Merging Fan-In)

builder = StateGraph(ShipmentRiskState)

# 1. Nodes Register Karein
builder.add_node("weather_agent", node_weather_risk)
builder.add_node("traffic_agent", node_traffic_risk)
builder.add_node("cost_agent", node_cost_risk)
builder.add_node("risk_assessment_agent", node_risk_assessment)

# 2. Parallel Branching (Fan-Out):
# START node se teeno risk agents ek saath parallel execution start karenge
builder.add_edge(START, "weather_agent")
builder.add_edge(START, "traffic_agent")
builder.add_edge(START, "cost_agent")

# 3. Merging (Fan-In):
# Teeno parallel risk agents merge honge main Risk Assessment Agent (Aggregator) par
builder.add_edge("weather_agent", "risk_assessment_agent")
builder.add_edge("traffic_agent", "risk_assessment_agent")
builder.add_edge("cost_agent", "risk_assessment_agent")

# 4. End Transition
builder.add_edge("risk_assessment_agent", END)

# Graph Compile Karein
shipment_graph = builder.compile()

In [23]:
# Block 7: Execution & Testing with Sample Shipment Data

# 1. Sample Shipment Input
sample_shipment = (
    "Shipment ID: SH-2026-9904\n"
    "Origin: Chicago, IL -> Destination: Miami, FL\n"
    "Cargo: Perishable Medical Supplies (Requires strict temperature control)\n"
    "Scheduled Transit: Severe winter storm advisory active on midwest routes, high holiday traffic expected."
)

# 2. Initial State Setup
initial_state = {
    "shipment_info": sample_shipment,
    "weather_risk": "",
    "traffic_risk": "",
    "cost_risk": "",
    "overall_assessment": "",
    "risk_reports": []
}

# 3. Graph Invoke
print("=" * 60)
print("RUNNING SHIPMENT RISK ASSESSMENT WORKFLOW...")
print("=" * 60)

final_output = shipment_graph.invoke(initial_state)

# 4. Print Accumulated Results
print(f"\n📦 SHIPMENT DETAILS:\n{final_output['shipment_info']}\n")
print("=" * 60)
print("ACCUMULATED RISK REPORTS & RECOMMENDATIONS")
print("=" * 60)

for report in final_output["risk_reports"]:
    print(report)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RUNNING SHIPMENT RISK ASSESSMENT WORKFLOW...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📦 SHIPMENT DETAILS:
Shipment ID: SH-2026-9904
Origin: Chicago, IL -> Destination: Miami, FL
Cargo: Perishable Medical Supplies (Requires strict temperature control)
Scheduled Transit: Severe winter storm advisory active on midwest routes, high holiday traffic expected.

ACCUMULATED RISK REPORTS & RECOMMENDATIONS
[COST RISK]
Shipment ID: SH-2026-9904
Origin: Chicago, IL -> Destination: Miami, FL
Cargo: Perishable Medical Supplies (Requires strict temperature control)
Scheduled Transit: Severe winter storm advisory active on midwest routes, high holiday traffic expected.

Shipment Data: 'Shipment ID: SH-2026-9904
Origin: Chicago
------------------------------------------------------------
[TRAFFIC RISK]
Output: Risk: High. Severe winter storm advisory and high holiday traffic expected.

Note: The example data is fictional and does not represent real-world scenarios.
------------------------------------------------------------
[WEATHER RISK]
I do not have access to real-time weather data